# 02 - Forecasting Experiments

## Related work and model selection

Cellular traffic forecasting has been approached with three broad families of
methods, each represented by one of the three models built in this notebook.

**Statistical/seasonal models.** Ferreira et al. [3] survey and benchmark
ARMA/ARIMA/SARIMA alongside neural approaches for network traffic forecasting
and note that seasonal ARIMA variants suit traffic with strong, regular
periodicity but become costly to fit as the seasonal period grows. Azari et
al. [2] compare ARIMA directly against LSTM on cellular traffic and find ARIMA
competitive on short, regular series, with LSTM's advantage growing with more
training data and finer granularity - motivating including *both* a
statistical and a deep-learning model here, not just one.

**Tree-based / feature-engineered ML models.** Kim [4] applies gradient
boosting ensembles to network traffic prediction and reports competitive
accuracy at a fraction of the training cost of deep sequence models, at the
expense of requiring the modeler to hand-engineer the lag structure rather
than letting the model discover it.

**Deep sequential models.** Santos et al. [1] train LSTM and GRU models on
*this same Milan dataset* for short-term mobile Internet traffic prediction
and report that recurrent models capture the daily/weekly structure well but
degrade during atypical periods not well represented in training data - a
finding directly relevant to the failure-case analysis in
`model_comparison.ipynb`.

**How `EDA.ipynb`'s findings informed model selection.** The EDA showed the
traffic series is strongly daily-periodic (the hour x weekday heatmap), has
autocorrelation that decays in a damped-periodic pattern over several days
while its partial autocorrelation is concentrated in the first few lags (the
ACF/PACF plots), and is stationary in levels (ADF test). That combination -
strong seasonality, short-range direct dependence, and literature evidence
that no single paradigm dominates - motivated three *structurally different*
models rather than three variants of one architecture:

1. **`SARIMAForecaster`** - a statistical model representing the daily
   seasonality explicitly (Fourier terms, motivated by the ACF's 144-lag
   periodicity and the heatmap's weekday/weekend contrast).
2. **`GBMForecaster`** - a tree-based model consuming the short-range
   dependence (PACF-motivated lags) and seasonality as engineered features.
3. **`LSTMForecaster`** - a recurrent model that learns temporal structure
   end-to-end from a raw window of history, rather than from hand-picked
   features.

**LSTM results vs. the literature.** Santos et al. [1] motivate LSTM/GRU as
the strongest architecture for this exact dataset, but the results in this
notebook (see "Final fit" below) don't reproduce that advantage: the LSTM
finishes last on every metric on all three squares, and is also several
times slower to train than gradient boosting (the exact ratio is computed in
the "Final fit" section, since wall-clock timing is noisier run-to-run than
the accuracy metrics). This isn't a contradiction of [1] so much as a
difference in scope - Santos et al. train on far more data and compute than
the six weeks of single-square history and the CPU-only, early-stopped,
deliberately small network used here (see "Hardware" in
`model_comparison.ipynb`). An architecture's advantage in the literature
and its advantage under this project's data and compute budget are two
different things, and only the second is what these results can speak to.

### References
[1] G. L. Santos, P. Rosati, T. Lynn, J. Kelner, D. Sadok, and P. T. Endo,
"Predicting short-term mobile Internet traffic from Internet activity using
recurrent neural networks," *Int. J. Netw. Manag.*, vol. 32, no. 3, e2191,
2022.
[2] A. Azari, P. Papapetrou, S. Denic, and G. Peters, "Cellular traffic
prediction and classification: a comparative evaluation of LSTM and ARIMA,"
in *Discovery Science (DS 2019)*, LNCS vol. 11828, Springer, Cham, 2019,
pp. 129-144.
[3] G. O. Ferreira, C. Ravazzi, F. Dabbene, G. Calafiore, and M. Fiore,
"Forecasting network traffic: a survey and tutorial with open-source
comparative evaluation," *IEEE Access*, vol. 11, 2023.
[4] H. Kim, "Network traffic prediction using gradient boosting ensemble
method," in *Proc. 2024 7th Artificial Intelligence and Cloud Computing Conf.
(AICCC)*, ACM, 2025, pp. 608-614.
[5] R. J. Hyndman and G. Athanasopoulos, *Forecasting: Principles and
Practice*, 3rd ed., OTexts, 2021, ch. 12 (Fourier terms for long seasonal
periods - see `forecasting/sarima_forecaster.py`).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "forecasting").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json
import time

import pandas as pd

from forecasting.data import SquareSeries, TUNING_TRAIN_END, TUNING_VAL_END, FINAL_TRAIN_END, TEST_START, TEST_END
from forecasting.sarima_forecaster import SARIMAForecaster
from forecasting.gbm_forecaster import GBMForecaster
from forecasting.lstm_forecaster import LSTMForecaster
from forecasting.evaluation import WalkForwardEvaluator
from forecasting.search import HyperparameterSearch
from forecasting.tracking import ExperimentTracker

COMBINED_PATH = ROOT / "data" / "processed" / "internet_traffic.parquet"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

with open(ROOT / "results" / "top_squares.json") as f:
    top_info = json.load(f)
TOP3 = top_info["top3_square_ids"]
TOP_SQUARE = TOP3[0]
print("Top-3 squares:", TOP3, "| tuning/search runs only on:", TOP_SQUARE)

tracker = ExperimentTracker(RESULTS_DIR / "experiment_log.csv")
evaluator = WalkForwardEvaluator()

Top-3 squares: [5161, 5059, 5259] | tuning/search runs only on: 5161


## Evaluation protocol

Every model is scored identically, one-step-ahead, via `WalkForwardEvaluator`:
at each target timestamp, the model only ever conditions on the *true* series
strictly before that point, never on its own prior prediction.

Data is split by time (10-minute resolution):

| Split | Range | Purpose |
|---|---|---|
| `tuning_train` | Nov 1 - Dec 8 | fit candidates during hyperparameter search |
| `tuning_val` | Dec 9 - Dec 15 (1 week) | select hyperparameters by validation RMSE |
| `final_train` | Nov 1 - Dec 15 | fit the chosen-hyperparameter model per square |
| `test` | **Dec 16 - Dec 22** (1 week) | held out; used for all reported plots/tables |

Hyperparameter search (`HyperparameterSearch`) runs once, on the
highest-traffic square only, and the winning configuration is reused on the
other two top squares - exhaustively re-tuning per square would triple the
search cost, and the three top squares share the same underlying
daily/weekly seasonal structure (`EDA.ipynb`), so hyperparameters picked
for temporal structure should transfer reasonably well even though absolute
traffic levels differ. Every trial, and the closing rationale for the
winning configuration, is logged by `ExperimentTracker` to
`results/experiment_log.csv`.

**Timing methodology.** Every training/prediction time in this notebook and
in `model_comparison.ipynb` is a single measured run, once per (model,
square) combination - not averaged or repeated. This is a simplification
given the compute budget available (see "Hardware" in
`model_comparison.ipynb`); with more time, repeating each measurement 3-5
times to also report variance would be the natural next step.

In [2]:
top_series = SquareSeries(TOP_SQUARE, COMBINED_PATH).load()

sarima_search = HyperparameterSearch(SARIMAForecaster, tracker)
t0 = time.time()
sarima_result = sarima_search.run(top_series, TUNING_TRAIN_END, top_series.loc[TUNING_TRAIN_END + pd.Timedelta(minutes=10):TUNING_VAL_END].index, square_id=TOP_SQUARE, model_name="SARIMA")
print(f"SARIMA search: {time.time()-t0:.1f}s")
print(sarima_result["rationale"])
pd.DataFrame(sarima_result["all_results"])

SARIMA search: 572.9s
Selected {'order': (2, 1, 2), 'n_harmonics': 3} - lowest validation RMSE (154.94) among 6 candidates.


,params,train_seconds,mae,mape,rmse
0,"{'order': (1, 0, 1), 'n_harmonics': 2}",2.029200,109.997241,10.119996,179.060068
1,"{'order': (1, 0, 1), 'n_harmonics': 3}",2.847686,109.264237,10.139513,178.144243
2,"{'order': (2, 0, 1), 'n_harmonics': 2}",5.291889,110.125575,10.143039,179.095126
3,"{'order': (2, 0, 1), 'n_harmonics': 3}",6.786410,109.361588,10.163999,178.163411
4,"{'order': (2, 1, 2), 'n_harmonics': 2}",3.247768,103.573602,8.097601,156.217419
5,"{'order': (2, 1, 2), 'n_harmonics': 3}",5.611193,102.766893,8.109953,154.939155


Order (2,1,2) with 3 Fourier harmonic pairs won, with validation RMSE 154.94
- clearly better than the two non-differenced orders (178.1-179.1) and
modestly better than its own d=1 sibling with 2 harmonics instead of 3
(156.22). The differenced candidates (d=1, RMSE 154.94/156.22) beat every
non-differenced (d=0) candidate (RMSE 178.1-179.1) by a wide, consistent
margin, despite `EDA.ipynb`'s ADF test finding the raw series stationary
in levels - a real but explainable tension: the ADF test only rules out a
*unit root*, not any benefit from differencing, and one difference evidently
still helps the ARIMA error process track slow drift within the 5,472-point
training window. Harmonic count (2 vs. 3) matters much less than
differencing here - at fixed d=1, 3 harmonics beats 2 by only 1.3 RMSE
(154.94 vs. 156.22), versus the ~22-25 RMSE gap between d=0 and d=1.

In [3]:
gbm_search = HyperparameterSearch(GBMForecaster, tracker)
val_index = top_series.loc[TUNING_TRAIN_END + pd.Timedelta(minutes=10):TUNING_VAL_END].index
t0 = time.time()
gbm_result = gbm_search.run(top_series, TUNING_TRAIN_END, val_index, square_id=TOP_SQUARE, model_name="GBM")
print(f"GBM search: {time.time()-t0:.1f}s")
print(gbm_result["rationale"])
pd.DataFrame(gbm_result["all_results"])

GBM search: 66.1s
Selected {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1} - lowest validation RMSE (158.28) among 4 candidates.


,params,train_seconds,mae,mape,rmse
0,"{'n_estimators': 100, 'max_depth': 3, 'learnin...",3.398694,101.861326,7.690756,158.280875
1,"{'n_estimators': 200, 'max_depth': 3, 'learnin...",6.645456,102.870391,7.662137,160.143524
2,"{'n_estimators': 200, 'max_depth': 4, 'learnin...",7.751358,101.366655,7.559657,159.347097
3,"{'n_estimators': 300, 'max_depth': 4, 'learnin...",10.997936,102.138056,7.609234,160.546820


The smallest candidate in the grid won (100 trees, max_depth=3,
learning_rate=0.1, validation RMSE 158.28) - every larger or deeper
candidate scored worse (159.3-160.5), not just no better. That's a sign the
engineered lag/calendar feature set (chosen from the EDA's PACF result) is
already informative enough that more trees or depth mainly add variance
rather than reduce bias here - the model needs better features, not more
capacity, and the features are fixed across this grid.

In [4]:
lstm_search = HyperparameterSearch(LSTMForecaster, tracker)
t0 = time.time()
lstm_result = lstm_search.run(top_series, TUNING_TRAIN_END, val_index, square_id=TOP_SQUARE, model_name="LSTM")
print(f"LSTM search: {time.time()-t0:.1f}s")
print(lstm_result["rationale"])
pd.DataFrame(lstm_result["all_results"])

LSTM search: 134.5s
Selected {'hidden_size': 32, 'num_layers': 1, 'lr': 0.001} - lowest validation RMSE (172.43) among 3 candidates.


,params,train_seconds,mae,mape,rmse
0,"{'hidden_size': 16, 'num_layers': 1, 'lr': 0.001}",21.511067,123.913671,12.042414,178.936431
1,"{'hidden_size': 32, 'num_layers': 1, 'lr': 0.001}",36.054224,121.913130,12.699054,172.429073
2,"{'hidden_size': 32, 'num_layers': 2, 'lr': 0.0...",68.951523,131.698262,12.681236,188.282308


hidden_size=32/num_layers=1/lr=1e-3 won (validation RMSE 172.43) over both
the smaller single-layer network (178.94) and the deeper two-layer network
(188.28 - the worst of the three despite being the most expressive
candidate and taking the longest to train). More capacity didn't translate
into a better validation score here, a sign that added depth increases
optimization difficulty faster than it increases useful capacity at this
training-set size (5,472 points). Even the winning LSTM configuration's
validation RMSE (172.43) is worse than both SARIMA's (154.94) and GBM's
(158.28) - the ranking that shows up in the final Dec 16-22 test results is
already visible here, on the validation week.

In [5]:
best_hp = {
    "SARIMA": sarima_result["best_params"],
    "GBM": gbm_result["best_params"],
    "LSTM": lstm_result["best_params"],
}
forecaster_classes = {"SARIMA": SARIMAForecaster, "GBM": GBMForecaster, "LSTM": LSTMForecaster}

# Final fit + one-step-ahead evaluation (Dec 16-22), all 3 top squares.
timing_rows = []
fitted_top_square_models = {}
for square_id in TOP3:
    series = SquareSeries(square_id, COMBINED_PATH).load()
    final_train = series.loc[:FINAL_TRAIN_END]
    test_index = series.loc[TEST_START:TEST_END].index

    for model_name, cls in forecaster_classes.items():
        model = cls(**best_hp[model_name])
        t0 = time.perf_counter()
        model.fit(final_train)
        train_seconds = time.perf_counter() - t0
        if square_id == TOP_SQUARE:
            fitted_top_square_models[model_name] = model  # for the epoch-count note below

        result = evaluator.run(model, series, test_index)
        metrics = {"mae": result["mae"], "mape": result["mape"], "rmse": result["rmse"]}

        tracker.log(
            model=model_name, params=best_hp[model_name], metrics=metrics,
            rationale=f"Final Dec16-22 evaluation on square {square_id}",
            square_id=square_id, phase="final",
            train_seconds=train_seconds, predict_seconds=result["predict_seconds"],
        )
        timing_rows.append({
            "square": square_id, "model": model_name,
            "train_seconds": round(train_seconds, 3),
            "predict_seconds": round(result["predict_seconds"], 3),
            **metrics,
        })

        result["predictions"].to_csv(RESULTS_DIR / f"predictions_{model_name.lower()}_{square_id}.csv", header=["prediction"])
        result["actuals"].to_csv(RESULTS_DIR / f"actuals_{square_id}.csv", header=["actual"])

        print(f"square={square_id} model={model_name:7s} MAE={metrics['mae']:.2f} MAPE={metrics['mape']:.2f} RMSE={metrics['rmse']:.2f} "
              f"train={train_seconds:.1f}s predict={result['predict_seconds']:.1f}s")

timing_df = pd.DataFrame(timing_rows)
timing_df.to_csv(RESULTS_DIR / "timing.csv", index=False)
print(f"\nLSTM early stopping on square {TOP_SQUARE}'s final fit: ran {fitted_top_square_models['LSTM'].epochs_run_} epoch(s) "
      f"(max 25, patience 4) - the SAME fit()-internal early-stopping mechanism used during hyperparameter "
      f"search above, just applied to final_train (6,480 points) instead of tuning_train (5,472 points), "
      f"consistent with how it's used everywhere else in this notebook.")

# LSTM/GBM training-time ratio (computed here since wall-clock timing is noisy).
lstm_train = timing_df[timing_df.model == "LSTM"].set_index("square")["train_seconds"]
gbm_train = timing_df[timing_df.model == "GBM"].set_index("square")["train_seconds"]
ratio = lstm_train / gbm_train
print(f"LSTM/GBM training-time ratio across the 3 squares: {ratio.min():.1f}x - {ratio.max():.1f}x")
timing_df

square=5161 model=SARIMA  MAE=82.57 MAPE=8.54 RMSE=124.26 train=5.8s predict=118.0s


square=5161 model=GBM     MAE=80.67 MAPE=8.64 RMSE=117.48 train=3.7s predict=9.0s


square=5161 model=LSTM    MAE=91.46 MAPE=9.72 RMSE=136.58 train=38.7s predict=2.2s


square=5059 model=SARIMA  MAE=71.36 MAPE=8.10 RMSE=98.15 train=16.9s predict=116.0s


square=5059 model=GBM     MAE=69.73 MAPE=7.33 RMSE=99.16 train=3.8s predict=9.2s


square=5059 model=LSTM    MAE=75.93 MAPE=8.48 RMSE=105.26 train=38.1s predict=2.5s


square=5259 model=SARIMA  MAE=69.79 MAPE=8.56 RMSE=96.49 train=17.2s predict=114.8s


square=5259 model=GBM     MAE=64.39 MAPE=7.23 RMSE=90.63 train=3.8s predict=8.9s


square=5259 model=LSTM    MAE=76.01 MAPE=9.05 RMSE=107.12 train=37.4s predict=2.9s

LSTM early stopping on square 5161's final fit: ran 25 epoch(s) (max 25, patience 4) - the SAME fit()-internal early-stopping mechanism used during hyperparameter search above, just applied to final_train (6,480 points) instead of tuning_train (5,472 points), consistent with how it's used everywhere else in this notebook.
LSTM/GBM training-time ratio across the 3 squares: 9.8x - 10.5x


,square,model,train_seconds,predict_seconds,mae,mape,rmse
0,5161,SARIMA,5.792,117.966,82.567241,8.536348,124.255727
1,5161,GBM,3.690,8.974,80.668467,8.639132,117.483657
2,5161,LSTM,38.669,2.177,91.459941,9.720637,136.583626
3,5059,SARIMA,16.875,116.042,71.359843,8.095990,98.150015
4,5059,GBM,3.763,9.168,69.728171,7.330903,99.161200
5,5059,LSTM,38.071,2.501,75.931211,8.476002,105.257291
6,5259,SARIMA,17.233,114.816,69.786836,8.556452,96.490553
7,5259,GBM,3.815,8.920,64.393131,7.230160,90.625046
8,5259,LSTM,37.362,2.884,76.006685,9.045989,107.121330


In [6]:
# Self-contained model description (Section 4-I/VI), read from BaseForecaster.describe().
for model_name in ["SARIMA", "GBM", "LSTM"]:
    desc = fitted_top_square_models[model_name].describe()
    print(f"=== {desc['name']} (selected hyperparameters: {desc['hyperparameters']}) ===")
    print(f"Structure:            {desc['structure']}")
    print(f"Input representation: {desc['input_representation']}")
    print(f"Preprocessing:        {desc['preprocessing']}")
    print(f"Training procedure:   {desc['training_procedure']}")
    print()

=== SARIMA (Fourier-augmented ARIMA) (selected hyperparameters: {'order': (2, 1, 2), 'n_harmonics': 3}) ===
Structure:            statsmodels SARIMAX, ARIMA order (2, 1, 2), with 3 Fourier harmonic pairs (period 144, the daily cycle) plus a weekend dummy as exogenous regressors, in place of a literal seasonal_order term (see module docstring).
Input representation: The full training series' levels, plus per-timestep exogenous Fourier sin/cos pairs (k=1..3) and an is_weekend dummy.
Preprocessing:        None - SARIMAX estimates its own error-process structure.
Training procedure:   Single maximum-likelihood fit (state-space form) on the training window; at prediction time, new true observations are appended to the fitted filter (refit=False) rather than re-estimating parameters, keeping one-step-ahead walk-forward evaluation cheap.

=== Gradient Boosting (lag + calendar features) (selected hyperparameters: {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1}) ===
Structure:      

This is the self-contained model description the assignment asks for, read
directly off the fitted model objects whose Dec 16-22 predictions are
reported above rather than a hand-written paraphrase that could drift from
what the code actually did. The three input representations are genuinely
different: SARIMA sees the raw level series plus a handful of Fourier/weekend
columns; GBM sees 13 hand-built columns per timestep (8 lags + 5 calendar
features); LSTM sees a raw 144-point window and builds its own
representation internally. Only the LSTM's preprocessing includes
normalization (z-score, fit on training data only) - SARIMA and GBM use the
traffic values in their native scale throughout, worth keeping in mind when
comparing error metrics directly against LSTM's. The LSTM's training
procedure line also reports how many of its (max 25) epochs actually ran
before early stopping triggered on this fit.

**GBM has the lowest MAE and MAPE on all three squares**, and the lowest
RMSE on two of three (SARIMA is marginally lower on square 5059's RMSE:
98.15 vs. GBM's 99.16, though GBM still wins that square's MAE/MAPE - GBM
has a few larger individual errors there that RMSE's squared penalty
punishes more than MAE does, without changing which model is preferable
overall). SARIMA is consistently second. **The LSTM is last on every metric
on every square**, despite being several times more expensive to train than
GBM (the exact ratio is printed above, from `train_seconds`) - the
accuracy-vs-compute trade-off resolves in GBM's favor across the board, not
just on the validation week.

This ranking is stable across squares with different absolute traffic levels
and different temporal profiles (`EDA.ipynb`: square 5259 has a flatter
early-week profile and a stronger weekday/weekend contrast than 5161 or
5059) - evidence the ranking reflects something about the models given this
data and compute budget, not an artifact of one square's traffic shape. See
`model_comparison.ipynb` for the full per-square tables, the 9 overlay
plots, and the failure-case analysis this result motivates.

`results/experiment_log.csv` now has every tuning trial plus these final
per-square runs, tagged by `phase` (`tuning`, `selected`, `final`) so
`model_comparison.ipynb` can filter it without re-deriving anything.